In [17]:
!pip install openai
from openai import OpenAI
import time # Used optionally for slight pause in streaming

print("Setup complete.")


Setup complete.


In [18]:
try:
    client = OpenAI(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio"  # Any string works for LM Studio local setup
    )
    print("\n✅ Successfully connected to LM Studio server.")
except Exception as e:
    print(f"\n❌ Error connecting to LM Studio: {e}")
    print("Please ensure your LM Studio server is running locally on port 1234 and the requested model is loaded.")



✅ Successfully connected to LM Studio server.


In [19]:
# 1. Initialize conversation history (this list will store all messages)
conversation_history = [
    {"role": "system", "content": "You are a helpful and friendly assistant. You remember our previous conversation."}
]

def chat_with_lm(client, history, new_user_query):
    """Sends the current conversation history to the model and handles streaming."""
    
    # Append the new user message to the history for context
    history.append({"role": "user", "content": new_user_query})

    print("\n--- Thinking... ---")
    
    try:
        stream = client.chat.completions.create(
            model="google/gemma-4-e2b",  # <-- CHANGED MODEL HERE
            messages=history,           # Send the ENTIRE history
            stream=True
        )

        full_response = ""
        print("🤖 Assistant:")
        for chunk in stream:
            content = chunk.choices[0].delta.content
            if content:
                print(content, end="", flush=True)
                full_response += content
        
        # 3. Store the assistant's response back into the history for future context
        assistant_response = {"role": "assistant", "content": full_response}
        history.append(assistant_response)
        print("\n------------------\n")
        return full_response

    except Exception as e:
        print(f"\nAn error occurred during API call: {e}")
        # If the API fails, remove the last user message so we don't break history
        history.pop() 
        return None



In [ ]:
if 'client' in locals():
    print("--- Starting Interactive Chat Session with Gemma ---\n")
    print("Type 'end' to exit the conversation.\n")
    
    while True:
        user_query = input("You: ")
        
        if user_query.lower() == 'end':
            print("\nChat ended. Goodbye!")
            break
        
        chat_with_lm(client, conversation_history, user_query)


--- Starting Interactive Chat Session with Gemma ---

User: hello Im Andrei

--- Thinking... ---
🤖 Assistant:
Hello Andrei! It's nice to meet you. 😊

How can I help you today?
------------------

